In [1]:
#!pip install -q faiss-cpu sentence-transformers ujson openai


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [3]:
import gc, torch, sys

# drop big globals if they exist
for name in ["model","tok","encoder","index","faiss_index"]:
    if name in globals(): del globals()[name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# sanity
# import subprocess, textwrap
# print(subprocess.getoutput("nvidia-smi"))

In [4]:
!kill -9 31256 310745

/bin/bash: line 1: kill: (31256) - No such process
/bin/bash: line 1: kill: (310745) - No such process


In [5]:
import subprocess, textwrap
print(subprocess.getoutput("nvidia-smi"))

Sun Nov  2 05:19:53 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.90.07              Driver Version: 550.90.07      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       On  |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P0             27W /   70W |     103MiB /  15360MiB |      4%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import torch, faiss, ujson as json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

/home/mmk2266/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
#!pip install -U huggingface_hub transformers

In [8]:
# ---- paths ----
EMB_DIR = Path("data/rag_embeddings")
INDEX_PATH = Path("data/rag.index")

In [9]:
# ---- Load FAISS index + metadata ----
index = faiss.read_index(str(INDEX_PATH))
meta = []
for jf in sorted(EMB_DIR.glob("*.jsonl")):
    with jf.open() as f:
        for line in f:
            meta.append(json.loads(line))
print(f"Loaded {len(meta)} chunks from {len(list(EMB_DIR.glob('*.jsonl')))} files.")

Loaded 5077 chunks from 23 files.


In [10]:
# ---- Load embedding model (for query encoding) ----
encoder = SentenceTransformer("Alibaba-NLP/gte-large-en-v1.5", trust_remote_code=True)

In [11]:
def retrieve(query, k=10):
    q_vec = encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, idxs = index.search(q_vec, k)
    results = []
    for score, i in zip(scores[0], idxs[0]):
        item = meta[int(i)]
        results.append({"score": float(score), **item})
    return results


In [12]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

In [13]:
MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
bnb_cfg = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_cfg,
    device_map="auto",
    low_cpu_mem_usage=True,
)

Loading checkpoint shards: 100%|██████████| 4/4 [01:25<00:00, 21.39s/it]


In [14]:
SYSTEM = (
  "Synthesize an answer using the provided context only.\n"
  "Do NOT copy sentences verbatim. Quote at most short phrases (<10 words).\n"
  "Cite sources inline as [DOC:doc_id]. If unknown, say 'Not found in the given context.'"
)

def build_context(chunks, max_chars=1500):
    blocks = []
    for c in chunks:
        text = (c.get("text_for_embedding") or c.get("text") or "")[:max_chars]
        blocks.append(f"<chunk doc_id='{c.get('doc_id')}' page='{c.get('page')}' type='{c.get('type')}'>\n{text}\n</chunk>")
    return "\n".join(blocks)

In [19]:
def generate_answer(query, context, max_new_tokens=512):
    SYSTEM = ("""You are a smart researcher. Ground every statement in the supplied <chunk> context only.
                If the context is insufficient, say 'Not found in the given context.'
                Cite sources inline as [DOC:doc_id, p:page]. Go through all the chunks from the context; quote <10 words.
                Prefer specific facts, numbers, definitions, algorithms, and caveats over generalities.
                You will see multiple blocks as context eg 'Chunk 1', 'Chunk 2', etc. Please go through all of them before giving an answer
                so that you won't miss anything!""")

    prompt = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n"
        f"{SYSTEM}\n"
        "<|eot_id|><|start_header_id|>user<|end_header_id|>\n"
        f"Question: {query}\n\n"
        "Use the <chunk> blocks below; paraphrase and cite as [DOC:doc_id].\n\n"
        f"{context}\n"
        "<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n"
    )

    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=0.5,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.15,
        no_repeat_ngram_size=8,
        eos_token_id=[tok.eos_token_id, tok.convert_tokens_to_ids("<|eot_id|>")],
        pad_token_id=tok.eos_token_id,
    )

    # only decode tokens after prompt
    gen_ids = out[:, inputs["input_ids"].shape[1]:]
    return tok.decode(gen_ids[0], skip_special_tokens=True).strip()

In [20]:
#query = "How do hyperparameters affect scaling?"
#query = "What do scaling laws say about how to choose model size and training compute?"
query = "What methods exist to accelerate token generation during inference without retraining the language model like Medusa?"
chunks = retrieve(query, k=10)
context = build_context(chunks, max_chars=1000) 

In [21]:
print(generate_answer(query, context, max_new_tokens=512))

Based on the provided context, there are several methods to accelerate token generation during inference without retraning the language model like Medusa:

1. **Speculative Decoding**: This involves using a draft model to generate multiple tokens in parallel, which are then verified by the main model (Leviathan et al., [2022]; Zhang et al., [2023b]; He et al., [2023]). However, this approach requires a separate draft model, which can be computationally expensive and may lead to distribution shift issues.
2. **Multiple Decoding Heads**: Adding multiple decoding heads on top of the base model to concurrently predict multiple tokens, as implemented in Medusa (Stern et al., [2018]). This approach allows for parallel processing and does not require a separate draft model.
3. **Tree-Based Attention Mechanism**: Constructing a tree-based attention mechanism to process multiple candidate tokens in parallel, as demonstrated in Medusa ([DOC:2401, p.7]).

These methods aim to reduce the sequentia

In [18]:
for i, ch in enumerate(chunks, 1):
    print(f"--- Chunk {i} | Score: {ch['score']:.3f} | Doc: {ch['doc_id']} | Page: {ch.get('page')} ---")
    print(ch.get("text_for_embedding", ch.get("text", ""))[:800])  # limit to 800 chars for readability
    print()


--- Chunk 1 | Score: 0.833 | Doc: 2401 | Page: 9 ---
[SECTION] Discussion [PAGE] 9
[PARAGRAPH]
In conclusion, MEDUSA enhances LLM inference speed by 2.3-2.8 times by equipping models with additional predictive decoding heads, allowing for generating multiple tokens simultaneously and bypassing the sequential decoding limitation. Key advantages of MEDUSA include its simplicity, parameter efficiency, and ease of integration into existing systems. MEDUSA avoids the need for specialized draft models. The typical acceptance scheme removes complications from rejection sampling while providing reasonable outputs. Our approach including two efficient training procedures, ensures high-quality output across various models and prompt types. We summarize the development of each technique and their impact on the speedup in Table 3.

--- Chunk 2 | Score: 0.768 | Doc: 2401 | Page: 16 ---
[SECTION] F. Additional Results on AlpacalEval Dataset [PAGE] 16
[FIGURE]
Caption: Figure 8. Speedup of various mo